# Lesson 4: 倍音と音色

**コンパニオンノートブック** — 詳しい解説は本文 Lesson 4 を参照してください。

## セットアップ

In [ ]:
# --- 最初に1回だけ実行 ---
import sys
try:
    import google.colab
    !pip install -q japanize-matplotlib
    !git clone -q https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
except ImportError:
    sys.path.append('..')

from audio_lib.notebook import setup_environment
setup_environment()

## このレッスンで学ぶこと

- 倍音列の概念を理解する
- 加算合成（サイン波の重ね合わせ）で音色を作る原理を学ぶ
- フーリエ級数の直感的な意味を理解する
- 倍音の構成が音色を決めることを実験で確かめる


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from audio_lib import sine_wave, additive_synth, AudioSignal
from audio_lib.notebook import play_sound, plot_waveform, plot_spectrum, plot_harmonics

## 4.1 倍音列とは

In [ ]:
f0 = 440  # 基本周波数
dur = 1.5

for k in range(1, 7):
    sig = sine_wave(f0 * k, dur)
    display(play_sound(sig, f"第{k}倍音: {f0 * k}Hz"))

## 4.2 加算合成で音色を作る

In [ ]:
f0 = 440

# 基本波だけ
display(play_sound(additive_synth(f0, {1: 1.0}), "第1倍音のみ"))

# 基本波 + 第2倍音
display(play_sound(additive_synth(f0, {1: 1.0, 2: 0.5}), "第1〜第2倍音"))

# 基本波 + 第2〜第3倍音
display(play_sound(additive_synth(f0, {1: 1.0, 2: 0.5, 3: 0.33}), "第1〜第3倍音"))

# 基本波 + 第2〜第5倍音
display(play_sound(additive_synth(f0, {1: 1.0, 2: 0.5, 3: 0.33, 4: 0.25, 5: 0.2}), "第1〜第5倍音"))

## 4.3 倍音の構成を変えて音色を作る

In [ ]:
harmonics = {}
for k in range(1, 11):
    harmonics[k] = 1.0

In [ ]:
# すべての倍音を等しい振幅で
sig_equal = additive_synth(f0, {k: 1.0 for k in range(1, 11)})
display(play_sound(sig_equal, "等振幅（10倍音）"))

# 奇数次倍音のみ（矩形波的）
sig_odd = additive_synth(f0, {k: 1.0/k for k in range(1, 21, 2)})
display(play_sound(sig_odd, "奇数次倍音のみ"))

# 高次倍音を強調（金属的な音）
sig_metal = additive_synth(f0, {k: float(k) for k in range(1, 11)})
display(play_sound(sig_metal, "高次倍音強調（金属的）"))

In [ ]:
plot_harmonics(sig_equal, max_freq=6000, title="等振幅")
plot_harmonics(sig_odd,   max_freq=6000, title="奇数次倍音のみ")
plot_harmonics(sig_metal, max_freq=6000, title="高次倍音強調")

In [ ]:
plot_spectrum(sig_equal, max_freq=6000, title="等振幅")
plot_spectrum(sig_odd,   max_freq=6000, title="奇数次倍音のみ")
plot_spectrum(sig_metal, max_freq=6000, title="高次倍音強調")

### クラリネット風の音

In [ ]:
# クラリネット風: 奇数次倍音が支配的、偶数次は弱い
sig_clarinet = additive_synth(f0, {
    1: 1.0, 2: 0.05, 3: 0.75, 4: 0.02, 5: 0.5,
    6: 0.01, 7: 0.3, 8: 0.01, 9: 0.15, 10: 0.01,
})

display(play_sound(sig_clarinet, "クラリネット風"))
plot_harmonics(sig_clarinet, max_freq=6000, title="クラリネット風の倍音構成")

### オルガン風の音

In [ ]:
# オルガン風: 基本波と低次倍音が均等に強い
sig_organ = additive_synth(f0, {
    1: 1.0, 2: 0.8, 3: 0.6, 4: 0.5, 5: 0.3, 6: 0.2, 8: 0.1,
})

display(play_sound(sig_organ, "オルガン風"))
plot_harmonics(sig_organ, max_freq=6000, title="オルガン風の倍音構成")

### ベル風の音（非整数倍音）

In [ ]:
# ベル風: 非整数倍の周波数を含む
sig_bell = additive_synth(f0, {
    1.0: 1.0, 2.0: 0.6, 2.76: 0.4, 3.95: 0.25, 5.14: 0.15, 6.58: 0.1,
})

display(play_sound(sig_bell, "ベル風"))
plot_spectrum(sig_bell, max_freq=4000, title="ベル風のスペクトラム")

## 4.5 additive_synth() の中身を見る

In [ ]:
sample_rate = 44100
dur = 2.0
t = np.linspace(0, dur, int(sample_rate * dur), endpoint=False)

# additive_synth() の中身と同じ処理
harmonics = {1: 1.0, 2: 0.5, 3: 0.3}  # レシピ
data = np.zeros_like(t)

for ratio, amp in harmonics.items():  # .items() でキーと値のペアを順に取り出す
    data += amp * np.sin(2 * np.pi * ratio * f0 * t)

# 最大振幅を 1.0 に正規化
data /= np.max(np.abs(data))

sig = AudioSignal(data, sample_rate)
display(play_sound(sig, "手書き加算合成"))

### 加算合成で3つの波形を再現する

In [ ]:
N = 20  # 足し合わせる倍音の数

# ノコギリ波
data_saw = np.zeros_like(t)
for k in range(1, N + 1):
    data_saw += ((-1.0) ** k / k) * np.sin(2 * np.pi * k * f0 * t)
data_saw *= -2.0 / np.pi

# 矩形波
data_sq = np.zeros_like(t)
for k in range(1, N * 2, 2):  # 奇数次のみ
    data_sq += (1.0 / k) * np.sin(2 * np.pi * k * f0 * t)
data_sq *= 4.0 / np.pi

# 三角波
data_tri = np.zeros_like(t)
for k in range(1, N * 2, 2):  # 奇数次のみ
    n_idx = (k - 1) // 2
    data_tri += ((-1.0) ** n_idx / k ** 2) * np.sin(2 * np.pi * k * f0 * t)
data_tri *= 8.0 / np.pi ** 2

# 波形を表示
show_samples = int(sample_rate * 2 / f0)
t_show = t[:show_samples] * 1000

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, (data, name) in zip(axes, [
    (data_saw, "ノコギリ波"),
    (data_sq, "矩形波"),
    (data_tri, "三角波"),
]):
    ax.plot(t_show, data[:show_samples])
    ax.set_ylabel(name)
    ax.set_ylim(-1.5, 1.5)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("時間 (ms)")
fig.suptitle(f"フーリエ級数による波形再現（倍音{N}個）", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
display(play_sound(AudioSignal(data_saw, sample_rate), "ノコギリ波（加算合成）"))
display(play_sound(AudioSignal(data_sq, sample_rate),  "矩形波（加算合成）"))
display(play_sound(AudioSignal(data_tri, sample_rate), "三角波（加算合成）"))

## 4.7 倍音の数と波形の関係を観察する

In [ ]:
harmonics_list = [1, 2, 3, 5, 10, 50]

fig, axes = plt.subplots(len(harmonics_list), 1, figsize=(12, 10), sharex=True)

for ax, N in zip(axes, harmonics_list):
    data = np.zeros_like(t)
    for k in range(1, N + 1):
        data += ((-1.0) ** k / k) * np.sin(2 * np.pi * k * f0 * t)
    data *= -2.0 / np.pi

    ax.plot(t_show, data[:show_samples])
    ax.set_ylabel(f"N={N}")
    ax.set_ylim(-1.3, 1.3)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("時間 (ms)")
fig.suptitle("ノコギリ波: 倍音の数による変化", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
for N in harmonics_list:
    data = np.zeros_like(t)
    for k in range(1, N + 1):
        data += ((-1.0) ** k / k) * np.sin(2 * np.pi * k * f0 * t)
    data *= -2.0 / np.pi
    display(play_sound(AudioSignal(data, sample_rate), f"ノコギリ波 N={N}"))

## 4.8 ギブス現象 — 不連続点の「ひげ」

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for ax, N, title in zip(axes, [20, 200], ["N=20", "N=200"]):
    data = np.zeros_like(t)
    for k in range(1, N * 2, 2):
        data += (1.0 / k) * np.sin(2 * np.pi * k * f0 * t)
    data *= 4.0 / np.pi

    ax.plot(t_show, data[:show_samples])
    ax.set_ylabel(title)
    ax.set_ylim(-1.5, 1.5)
    ax.axhline(y=1.0, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=-1.0, color='r', linestyle='--', alpha=0.5)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("時間 (ms)")
fig.suptitle("ギブス現象: 矩形波の不連続点付近のオーバーシュート", fontsize=14)
plt.tight_layout()
plt.show()

### 帯域制限付き加算合成

In [ ]:
# 帯域制限付きノコギリ波: ナイキスト周波数以下の倍音のみ
nyquist = sample_rate / 2.0
max_harmonic = int(nyquist / f0)

data_bl = np.zeros_like(t)
for k in range(1, max_harmonic + 1):
    data_bl += ((-1.0) ** k / k) * np.sin(2 * np.pi * k * f0 * t)
data_bl *= -2.0 / np.pi

print(f"基本周波数: {f0}Hz")
print(f"ナイキスト周波数: {nyquist}Hz")
print(f"使用する倍音の数: {max_harmonic}個")
print(f"最高倍音の周波数: {max_harmonic * f0}Hz")

In [ ]:
# 低い音と高い音で倍音の数を比較
for freq in [100, 440, 2000, 8000]:
    n = int(nyquist / freq)
    print(f"{freq:>5}Hz → 倍音 {n}個（最高 {n * freq}Hz）")